# QuantUI-local — Quantum Chemistry Calculator

Run PySCF quantum chemistry calculations directly in this notebook.

**How to use:**
1. Select or enter a molecule in **Molecule Input**
2. Choose a method and basis set in **Calculation Setup**
3. Click **Run Calculation** — results appear below
4. Optionally add results to **Compare** or **Export** a standalone script

> **Platform note:** PySCF requires Linux, macOS, or WSL.  \
> Windows users: `apptainer run quantui-local.sif`


In [ ]:
# Environment check — verifies correct conda environment.
# Tagged skip-execution and remove-input so it is hidden in Voilà.
import sys as _sys
_env = _sys.prefix
if "quantui" not in _env.lower():
    print(f"Warning: active environment may not be quantui-local")
    print(f"Active: {_env}")
    print("Run: conda activate quantui-local")


In [ ]:
import threading
import ipywidgets as widgets
from IPython.display import display, HTML

import quantui
from quantui import (
    Molecule, parse_xyz_input,
    MOLECULE_LIBRARY, SUPPORTED_METHODS, SUPPORTED_BASIS_SETS,
    DEFAULT_METHOD, DEFAULT_BASIS, DEFAULT_CHARGE, DEFAULT_MULTIPLICITY,
    session_can_handle, get_session_resources,
    PUBCHEM_AVAILABLE, VISUALIZATION_AVAILABLE, ASE_AVAILABLE,
    QUICK_START_TEMPLATES,
)

# Optional — degrade gracefully if unavailable
try:
    from quantui import run_in_session, SessionResult
    PYSCF_AVAILABLE = True
except (ImportError, AttributeError):
    PYSCF_AVAILABLE = False

try:
    from quantui import student_friendly_fetch
except (ImportError, AttributeError):
    student_friendly_fetch = None

try:
    from quantui import display_molecule
except (ImportError, AttributeError):
    display_molecule = None

try:
    from quantui import preoptimize
    PREOPT_AVAILABLE = True
except (ImportError, AttributeError):
    PREOPT_AVAILABLE = False

# Mutable session state shared across all callbacks
_state = {"molecule": None, "last_result": None, "results": []}

# ── Global app styles ─────────────────────────────────────────────────────────
# Permanent — not toggled by dark mode (dark mode uses filter inversion on top).
display(HTML("""<style>
/* System font stack ---------------------------------------------------- */
body, p, span, li, td, th, label, input, select, textarea, blockquote,
.jp-OutputArea-output, .widget-html-content, .widget-label-basic,
.widget-label {
    font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", system-ui,
                 Roboto, "Helvetica Neue", Arial, sans-serif !important;
    -webkit-font-smoothing: antialiased;
}

/* App title (h1 in the markdown cell) ---------------------------------- */
h1 {
    font-size: 20px !important;
    font-weight: 700 !important;
    color: #1e293b !important;
    letter-spacing: -0.01em !important;
    margin: 10px 0 4px !important;
    border-bottom: none !important;
}

/* Section headers ------------------------------------------------------- */
/* Overrides the inline h3 styles used in every display cell. */
h3 {
    font-size: 11px !important;
    font-weight: 700 !important;
    letter-spacing: 0.09em !important;
    text-transform: uppercase !important;
    color: #64748b !important;
    margin: 24px 0 10px !important;
    padding-bottom: 5px !important;
    border-bottom: 1px solid #e2e8f0 !important;
}

/* Rounded corners on inputs, dropdowns, and buttons -------------------- */
.widget-text input, .widget-textarea textarea {
    border-color: #d1d5db !important;
    border-radius: 5px !important;
}
.widget-dropdown select { border-radius: 5px !important; }
.widget-button, .widget-toggle-button { border-radius: 5px !important; }
</style>"""))

# ── Dark mode toggle ─────────────────────────────────────────────────────────
# Uses CSS filter invert+hue-rotate on the html element so it works with all
# inline-styled elements. canvas/img/iframe are re-inverted to keep their
# original appearance (e.g. the py3Dmol 3D viewer).
_DARK_CSS = (
    "<style>"
    "html { filter: invert(1) hue-rotate(180deg) !important; }"
    "canvas, img, iframe, video "
    "{ filter: invert(1) hue-rotate(180deg) !important; }"
    "</style>"
)

_theme_style = widgets.Output(
    layout=widgets.Layout(height="0px", overflow="hidden", margin="0", padding="0")
)

dark_mode_btn = widgets.ToggleButton(
    value=False,
    description=" Dark mode",
    icon="moon-o",
    tooltip="Toggle dark / light mode",
    layout=widgets.Layout(width="136px", height="30px"),
)


def _toggle_theme(change):
    _theme_style.clear_output()
    if change["new"]:
        dark_mode_btn.icon = "sun-o"
        dark_mode_btn.description = " Light mode"
        with _theme_style:
            display(HTML(_DARK_CSS))
    else:
        dark_mode_btn.icon = "moon-o"
        dark_mode_btn.description = " Dark mode"


dark_mode_btn.observe(_toggle_theme, names="value")

display(widgets.HBox(
    [dark_mode_btn],
    layout=widgets.Layout(justify_content="flex-end", margin="0 0 4px"),
))
display(_theme_style)

# ── Status panel ────────────────────────────────────────────────────────────
_cores, _mem_gb = get_session_resources()
_mem = f"{_mem_gb} GB" if _mem_gb is not None else "unknown"


def _ok(flag, extra=""):
    tick = '<span style="color:#22c55e">&#10003;</span>'
    cross = '<span style="color:#ef4444">&#10007;</span>'
    return (tick if flag else cross) + (" " + extra if extra else "")


_items = [
    ("PySCF (calculations)", _ok(PYSCF_AVAILABLE,
        "" if PYSCF_AVAILABLE else "&mdash; Linux / macOS / WSL required")),
    ("ASE (structure I/O, opt.)", _ok(ASE_AVAILABLE)),
    ("PubChem search", _ok(PUBCHEM_AVAILABLE)),
    ("3D viewer (py3Dmol)", _ok(VISUALIZATION_AVAILABLE)),
    ("CPU cores / Memory", f"<b>{_cores}</b> cores / <b>{_mem}</b>"),
]
_rows = "".join(
    f'<tr><td style="padding:3px 16px 3px 0;color:#64748b;font-size:13px">{k}</td>'
    f'<td style="font-size:13px">{v}</td></tr>'
    for k, v in _items
)
display(HTML(
    f'<div style="background:#f8fafc;border:1px solid #e2e8f0;border-left:4px solid #3b82f6;'
    f'padding:12px 16px;border-radius:6px;margin:4px 0 8px">'
    f'<span style="font-weight:600;font-size:14px;color:#1e293b">'
    f"QuantUI-local {quantui.__version__}</span>"
    f'<table style="margin-top:8px;border-collapse:collapse">{_rows}</table>'
    f"</div>"
))


In [ ]:
import io

from quantui.progress import StepProgress

# ── Shared output widgets ────────────────────────────────────────────────────
mol_info_html = widgets.HTML(
    value='<i style="color:#888">No molecule loaded yet.</i>'
)
mol_summary_compact = widgets.HTML(value="")
viz_output    = widgets.Output(layout=widgets.Layout(min_height="50px"))
run_output    = widgets.Output(
    layout=widgets.Layout(
        border="1px solid #c0ccd8", min_height="80px", max_height="400px",
        padding="8px", overflow_y="auto",
    )
)
with run_output:
    display(HTML(
        '<p style="color:#999;font-style:italic;font-size:13px;margin:2px 0">'
        "No calculation run yet. PySCF output and any errors will appear here."
        "</p>"
    ))
result_output     = widgets.Output()
comparison_output = widgets.Output()
notes_output      = widgets.Output()

# ── Step indicator ────────────────────────────────────────────────────────────
step_progress = StepProgress(["Choose molecule", "Set method", "Run", "Results"])
step_progress.start(0)

# ── Calculation setup (defined here so _set_molecule can update them) ────────
method_dd = widgets.Dropdown(
    options=SUPPORTED_METHODS, value=DEFAULT_METHOD,
    description="Method:", style={"description_width": "100px"},
    layout=widgets.Layout(width="260px"),
)
basis_dd = widgets.Dropdown(
    options=SUPPORTED_BASIS_SETS, value=DEFAULT_BASIS,
    description="Basis Set:", style={"description_width": "100px"},
    layout=widgets.Layout(width="260px"),
)
charge_si = widgets.BoundedIntText(
    value=DEFAULT_CHARGE, min=-10, max=10,
    description="Charge:", style={"description_width": "100px"},
    layout=widgets.Layout(width="190px"),
)
mult_si = widgets.BoundedIntText(
    value=DEFAULT_MULTIPLICITY, min=1, max=10,
    description="Multiplicity:", style={"description_width": "100px"},
    layout=widgets.Layout(width="190px"),
)
preopt_cb = widgets.Checkbox(
    value=False,
    description="Pre-optimize geometry (fast LJ force-field)",
    disabled=not PREOPT_AVAILABLE,
    layout=widgets.Layout(width="400px"),
)

# ── Context-help buttons (next to Method and Basis dropdowns) ────────────────
method_help_btn = widgets.Button(
    description="?", button_style="",
    layout=widgets.Layout(width="28px", height="28px"),
    tooltip="RHF vs UHF — opens Help tab",
)
basis_help_btn = widgets.Button(
    description="?", button_style="",
    layout=widgets.Layout(width="28px", height="28px"),
    tooltip="Choosing a basis set — opens Help tab",
)

# ── Run widgets ──────────────────────────────────────────────────────────────
run_btn = widgets.Button(
    description="Run Calculation", button_style="success", icon="play",
    disabled=True, layout=widgets.Layout(width="200px", height="36px"),
)
run_status = widgets.Label()

# ── Log clear button ─────────────────────────────────────────────────────────
log_clear_btn = widgets.Button(
    description="Clear", button_style="", icon="times",
    layout=widgets.Layout(width="90px", height="26px"),
    tooltip="Clear calculation output",
)

def _clear_log(btn):
    run_output.clear_output()

log_clear_btn.on_click(_clear_log)

# ── Comparison / export widgets ───────────────────────────────────────────────
accumulate_btn = widgets.Button(
    description="Add to Comparison", button_style="info", icon="plus",
    disabled=True, layout=widgets.Layout(width="190px"),
)
clear_btn = widgets.Button(
    description="Clear", button_style="warning", icon="trash",
    layout=widgets.Layout(width="100px"),
)
export_btn = widgets.Button(
    description="Export Script", button_style="", icon="download",
    disabled=True, layout=widgets.Layout(width="160px"),
)
export_status = widgets.Label()


# ── Thread-safe log capture ───────────────────────────────────────────────────
class _LogCapture:
    '''Write PySCF output to an Output widget and capture it to a buffer.'''
    def __init__(self, output_widget):
        self._w = output_widget
        self._buf = io.StringIO()

    def write(self, text: str) -> None:
        if text:
            self._w.append_stdout(text)
            self._buf.write(text)

    def flush(self) -> None:
        pass

    def getvalue(self) -> str:
        return self._buf.getvalue()


# Placeholders — overwritten by later cells once they execute.
_refresh_results_browser = lambda: None  # noqa: E731
_show_help_topic = lambda topic: None  # noqa: E731


# ── Callbacks ─────────────────────────────────────────────────────────────────

def _set_molecule(mol, label=""):
    '''Update shared state and refresh dependent widgets.'''
    _state["molecule"] = mol
    run_btn.disabled    = False
    export_btn.disabled = False

    try:
        n_e = mol.get_electron_count()
        e_str = f"{n_e} electrons"
    except Exception:
        e_str = ""

    _lbl = f'<br><small style="color:#777">{label}</small>' if label else ""
    _summary = (
        f'<b style="font-size:15px">{mol.get_formula()}</b>'
        f'&ensp;<span style="color:#555;font-size:13px">'
        f"{len(mol.atoms)} atoms"
        + (f" &bull; {e_str}" if e_str else "")
        + f" &bull; charge {mol.charge} &bull; mult {mol.multiplicity}"
        + f"</span>{_lbl}"
    )
    mol_info_html.value = _summary
    mol_summary_compact.value = (
        f'<div style="background:#f0f9ff;border:1px solid #bae6fd;'
        f'border-radius:6px;padding:7px 14px;font-size:14px;display:inline-block">'
        f"{_summary}</div>"
    )

    charge_si.value = mol.charge
    mult_si.value   = mol.multiplicity
    if mol.multiplicity > 1 and method_dd.value == "RHF":
        method_dd.value = "UHF"

    viz_output.clear_output(wait=True)
    if display_molecule is not None:
        with viz_output:
            display_molecule(mol)

    _update_notes()

    # Advance step indicator — but not during a running calculation (e.g. preopt)
    if step_progress._states[2] != "active":
        if step_progress._states[2] in ("done", "fail"):
            # Fresh molecule after a completed run — reset indicator
            step_progress.reset()
        step_progress.complete(0)
        step_progress.start(1)

    # Collapse the molecule input panel to the compact summary + 3D viewer
    mol_input_container.children = [mol_input_collapsed, viz_output]


def _format_result(r):
    _conv   = "Yes" if r.converged else "No (treat results with caution)"
    _cc     = "green" if r.converged else "#c00"
    _gap    = (
        f"{r.homo_lumo_gap_ev:.4f} eV"
        if r.homo_lumo_gap_ev is not None else "N/A"
    )
    _rows = "".join(
        f'<tr>'
        f'<td style="padding:3px 18px 3px 0;color:#444">{k}</td>'
        f'<td style="color:{vc}">{v}</td>'
        f"</tr>"
        for k, v, vc in [
            ("Total energy",
             f"{r.energy_hartree:.8f} Ha &ensp;({r.energy_ev:.4f} eV)", "#000"),
            ("HOMO-LUMO gap",   _gap, "#000"),
            ("SCF converged",   _conv, _cc),
            ("SCF iterations",  str(r.n_iterations), "#000"),
        ]
    )
    return (
        f'<div style="background:#f0fff0;border-left:4px solid #4CAF50;'
        f'padding:10px 14px;border-radius:4px;margin:6px 0">'
        f"<b>{r.formula} &mdash; {r.method}/{r.basis}</b>"
        f'<table style="margin-top:8px;font-size:14px;border-collapse:collapse">'
        f"{_rows}</table></div>"
    )


def _do_run(btn):
    mol = _state["molecule"]
    if mol is None:
        run_status.value = "Load a molecule first."
        return
    run_btn.disabled = True
    run_status.value = "Starting..."
    run_output.clear_output()
    result_output.clear_output()

    # Advance step indicator: method confirmed → running
    step_progress.complete(1)
    step_progress.start(2)

    def _thread():
        log = _LogCapture(run_output)
        try:
            calc_mol = mol
            if preopt_cb.value and PREOPT_AVAILABLE:
                run_status.value = "Pre-optimizing..."
                calc_mol, _rmsd = preoptimize(mol)
                _set_molecule(calc_mol, f"Geometry pre-optimized (LJ, RMSD={_rmsd:.3f} Å)")

            run_status.value = "Calculating..."
            result = run_in_session(
                molecule=calc_mol,
                method=method_dd.value,
                basis=basis_dd.value,
                progress_stream=log,
            )

            _state["last_result"] = result
            accumulate_btn.disabled = False

            result_output.append_display_data(HTML(_format_result(result)))
            run_status.value = "Done."

            # Advance step indicator: run complete → results ready
            step_progress.complete(2)
            step_progress.complete(3)

            try:
                from quantui import save_result
                save_result(result, pyscf_log=log.getvalue())
                _refresh_results_browser()
            except Exception:
                pass

        except ImportError:
            log.write(
                "PySCF is not available in this environment.\n"
                "PySCF requires Linux, macOS, or WSL.\n"
                "On Windows: use the Apptainer container.\n"
                "  apptainer run quantui-local.sif\n"
            )
            run_status.value = "PySCF unavailable."
            step_progress.fail(2, "PySCF unavailable")

        except Exception as exc:
            import traceback
            log.write(f"Error: {exc}\n\n{traceback.format_exc()}")
            run_status.value = "Error — see Calculation Output below."
            step_progress.fail(2, str(exc)[:60])

        finally:
            run_btn.disabled = False

    threading.Thread(target=_thread, daemon=True).start()

run_btn.on_click(_do_run)


def _update_notes(change=None):
    notes_output.clear_output(wait=True)
    mol = _state["molecule"]
    if mol is None:
        return
    try:
        from quantui import PySCFCalculation
        calc  = PySCFCalculation(mol, method=method_dd.value, basis=basis_dd.value)
        notes = calc.get_educational_notes()
        if notes:
            safe = (
                notes
                .replace("**", "<b>", 1)
                .replace("**", "</b>", 1)
                .replace("\n\n", "<br><br>")
            )
            with notes_output:
                display(HTML(
                    '<div style="background:#fffbf0;padding:8px 12px;'
                    'border-radius:4px;font-size:13px;margin-top:6px">'
                    + safe + "</div>"
                ))
    except Exception:
        pass

method_dd.observe(_update_notes, names="value")
basis_dd.observe(_update_notes, names="value")


def _do_accumulate(btn):
    r = _state["last_result"]
    if r is None:
        return
    _state["results"].append(r)
    _refresh_comparison()

accumulate_btn.on_click(_do_accumulate)


def _refresh_comparison():
    from quantui import summary_from_session_result, comparison_table_html
    comparison_output.clear_output(wait=True)
    results = _state["results"]
    if not results:
        return
    summaries = [summary_from_session_result(r) for r in results]
    with comparison_output:
        display(HTML(comparison_table_html(summaries)))
        if len(summaries) > 1:
            try:
                from quantui import plot_comparison
                plot_comparison(summaries)
            except Exception:
                pass


def _do_clear(btn):
    _state["results"].clear()
    comparison_output.clear_output()

clear_btn.on_click(_do_clear)


def _do_export(btn):
    mol = _state["molecule"]
    if mol is None:
        export_status.value = "Load a molecule first."
        return
    try:
        from quantui import PySCFCalculation
        from pathlib import Path
        calc  = PySCFCalculation(mol, method=method_dd.value, basis=basis_dd.value)
        fname = f"{mol.get_formula()}_{method_dd.value}_{basis_dd.value}.py"
        calc.generate_calculation_script(Path(fname))
        export_status.value = f"Saved: {fname}"
    except Exception as exc:
        export_status.value = f"Error: {exc}"

export_btn.on_click(_do_export)


def _on_method_help(btn):
    _show_help_topic("method")

def _on_basis_help(btn):
    _show_help_topic("basis_set")

method_help_btn.on_click(_on_method_help)
basis_help_btn.on_click(_on_basis_help)


In [ ]:
# ── Preset Library ─────────────────────────────────────────────────────────
_preset_opts = ["(select a molecule)"] + list(MOLECULE_LIBRARY.keys())
preset_dd = widgets.Dropdown(
    options=_preset_opts, value="(select a molecule)",
    description="Molecule:", style={"description_width": "90px"},
    layout=widgets.Layout(width="320px"),
)

def _load_preset(change):
    name = change["new"]
    if name.startswith("("):
        return
    d = MOLECULE_LIBRARY[name]
    _set_molecule(
        Molecule(
            atoms=d["atoms"], coordinates=d["coordinates"],
            charge=d["charge"], multiplicity=d["multiplicity"],
        ),
        d["description"],
    )

preset_dd.observe(_load_preset, names="value")

# ── XYZ Input ──────────────────────────────────────────────────────────────
xyz_area = widgets.Textarea(
    placeholder=(
        "Paste XYZ coordinates (symbol  x  y  z):\n"
        "O  0.000  0.000  0.000\n"
        "H  0.757  0.587  0.000\n"
        "H -0.757  0.587  0.000"
    ),
    layout=widgets.Layout(width="440px", height="130px"),
)
xyz_btn = widgets.Button(description="Load XYZ", button_style="info", icon="upload")
xyz_msg = widgets.Label()

def _load_xyz(btn):
    try:
        mol = parse_xyz_input(xyz_area.value.strip())
        _set_molecule(mol, "Loaded from XYZ input")
        xyz_msg.value = ""
    except Exception as exc:
        xyz_msg.value = f"Parse error: {exc}"

xyz_btn.on_click(_load_xyz)

# ── PubChem Search ─────────────────────────────────────────────────────────
pubchem_txt = widgets.Text(
    placeholder="name or SMILES  (e.g. aspirin, caffeine, CC(=O)O)",
    layout=widgets.Layout(width="380px"),
)
pubchem_btn = widgets.Button(
    description="Search", button_style="info", icon="search",
    disabled=not PUBCHEM_AVAILABLE,
    layout=widgets.Layout(width="100px"),
)
pubchem_msg = widgets.Label(
    value="" if PUBCHEM_AVAILABLE else "PubChem unavailable — check internet connection"
)

def _search_pubchem(btn):
    query = pubchem_txt.value.strip()
    if not query:
        pubchem_msg.value = "Enter a molecule name or SMILES."
        return
    if student_friendly_fetch is None:
        pubchem_msg.value = "PubChem module not available."
        return
    pubchem_msg.value = f'Searching for "{query}"...'
    pubchem_btn.disabled = True

    def _do():
        try:
            mol = student_friendly_fetch(query)
            _set_molecule(mol, f"PubChem: {query}")
            pubchem_msg.value = f"Loaded {mol.get_formula()} from PubChem."
        except Exception as exc:
            pubchem_msg.value = f"Not found: {exc}"
        finally:
            pubchem_btn.disabled = False

    threading.Thread(target=_do, daemon=True).start()

pubchem_btn.on_click(_search_pubchem)

# ── Assemble input tab ─────────────────────────────────────────────────────
_hint = '<p style="margin:4px 0 8px;color:#666;font-size:13px">'
tab_preset = widgets.VBox([
    widgets.HTML(_hint + "Choose from 20+ curated educational molecules.</p>"),
    preset_dd,
])
tab_xyz = widgets.VBox([
    widgets.HTML(_hint + "Paste XYZ coordinates (element x y z, one atom per line).</p>"),
    xyz_area,
    widgets.HBox([xyz_btn, xyz_msg]),
])
tab_pubchem = widgets.VBox([
    widgets.HTML(_hint + "Search by name or SMILES. Requires internet connection.</p>"),
    widgets.HBox([pubchem_txt, pubchem_btn]),
    pubchem_msg,
])

input_tab = widgets.Tab(children=[tab_preset, tab_xyz, tab_pubchem])
for _i, _t in enumerate(["Preset Library", "XYZ Input", "PubChem Search"]):
    input_tab.set_title(_i, _t)

# ── Collapsible molecule input container ──────────────────────────────────
mol_input_expanded = widgets.VBox([
    widgets.HTML('<h3 style="margin:8px 0 6px">Molecule Input</h3>'),
    input_tab,
])

# Change button — re-expands the input panel
change_mol_btn = widgets.Button(
    description="Change", button_style="", icon="pencil",
    layout=widgets.Layout(width="100px", height="32px"),
    tooltip="Re-expand the molecule input panel",
)

# Collapsed view: compact summary pill + Change button
mol_input_collapsed = widgets.HBox(
    [mol_summary_compact, change_mol_btn],
    layout=widgets.Layout(align_items="center", gap="12px", padding="6px 0"),
)

# Container starts expanded; _set_molecule() collapses it after first load.
mol_input_container = widgets.VBox(
    [mol_input_expanded, mol_info_html, viz_output],
    layout=widgets.Layout(margin="0 0 4px 0"),
)

def _expand_mol_input(btn):
    mol_input_container.children = [mol_input_expanded, mol_info_html, viz_output]

change_mol_btn.on_click(_expand_mol_input)


In [ ]:
calc_setup_panel = widgets.VBox([
    widgets.HTML('<h3 style="margin:14px 0 6px">Calculation Setup</h3>'),
    widgets.HBox([
        widgets.VBox([
            widgets.HBox(
                [method_dd, method_help_btn],
                layout=widgets.Layout(align_items="center", gap="4px"),
            ),
            widgets.HBox(
                [basis_dd, basis_help_btn],
                layout=widgets.Layout(align_items="center", gap="4px"),
            ),
        ]),
        widgets.HTML("&ensp;&ensp;"),
        widgets.VBox([charge_si, mult_si]),
    ]),
    preopt_cb,
    notes_output,
])


In [ ]:
run_panel = widgets.VBox([
    widgets.HTML(
        '<h3 style="margin:14px 0 6px">Run Calculation</h3>'
        '<p style="color:#555;font-size:13px;margin:0 0 8px">PySCF runs in this kernel. '
        "Output appears live below. Large molecules or high-accuracy basis sets may take "
        "several minutes on a laptop.</p>"
    ),
    widgets.HBox([run_btn, run_status]),
    widgets.HBox(
        [
            widgets.HTML(
                '<span style="font-size:13px;font-weight:600;color:#444">'
                "Calculation Output</span>"
            ),
            log_clear_btn,
        ],
        layout=widgets.Layout(align_items="center", justify_content="space-between",
                              margin="10px 0 4px", max_width="460px"),
    ),
    run_output,
])


In [ ]:
results_panel = widgets.VBox([
    widgets.HTML('<h3 style="margin:14px 0 6px">Results</h3>'),
    result_output,
])


In [ ]:
import json
import os
import subprocess
from pathlib import Path

# ── Past Results browser ──────────────────────────────────────────────────────
past_dd = widgets.Dropdown(
    description="Load:",
    options=[("(no saved results)", "")],
    style={"description_width": "50px"},
    layout=widgets.Layout(width="500px"),
)
past_refresh_btn = widgets.Button(
    description="Refresh", button_style="", icon="refresh",
    layout=widgets.Layout(width="100px"),
    tooltip="Rescan the results directory",
)
open_folder_btn = widgets.Button(
    description="Open folder", button_style="", icon="folder-open",
    layout=widgets.Layout(width="130px"),
    tooltip="Open the results directory in your file manager",
)
results_path_lbl = widgets.Label()
past_output = widgets.Output()


def _get_results_dir() -> Path:
    from quantui.results_storage import _default_results_dir
    return _default_results_dir().resolve()


def _format_past_result(data: dict) -> str:
    _conv = "Yes" if data.get("converged") else "No (treat results with caution)"
    _cc   = "green" if data.get("converged") else "#c00"
    _gap  = (
        f"{data['homo_lumo_gap_ev']:.4f} eV"
        if data.get("homo_lumo_gap_ev") is not None else "N/A"
    )
    _rows = "".join(
        f'<tr>'
        f'<td style="padding:3px 18px 3px 0;color:#444">{k}</td>'
        f'<td style="color:{vc}">{v}</td>'
        f"</tr>"
        for k, v, vc in [
            ("Total energy",
             f"{data['energy_hartree']:.8f} Ha &ensp;({data['energy_ev']:.4f} eV)", "#000"),
            ("HOMO-LUMO gap",   _gap, "#000"),
            ("SCF converged",   _conv, _cc),
            ("SCF iterations",  str(data.get("n_iterations", "?")), "#000"),
        ]
    )
    ts = data.get("timestamp", "")
    return (
        f'<div style="background:#f0fff0;border-left:4px solid #4CAF50;'
        f'padding:10px 14px;border-radius:4px;margin:6px 0">'
        f'<b>{data["formula"]} &mdash; {data["method"]}/{data["basis"]}</b>'
        f'&ensp;<small style="color:#777">{ts}</small>'
        f'<table style="margin-top:8px;font-size:14px;border-collapse:collapse">'
        f"{_rows}</table></div>"
    )


def _refresh_results_browser():
    '''Repopulate the past-results dropdown. Called on startup and after each run.'''
    try:
        from quantui import list_results, load_result
    except ImportError:
        return
    results_path_lbl.value = str(_get_results_dir())
    dirs = list_results()
    if not dirs:
        past_dd.options = [("(no saved results)", "")]
        return
    options = []
    for d in dirs:
        try:
            data = load_result(d)
            ts    = data.get("timestamp", d.name)
            label = f"{ts}  ·  {data['formula']}  {data['method']}/{data['basis']}"
            options.append((label, str(d)))
        except Exception:
            pass
    past_dd.options = options if options else [("(no saved results)", "")]


def _load_past_result(change):
    path_str = change["new"]
    if not path_str:
        past_output.clear_output()
        return
    past_output.clear_output(wait=True)
    try:
        from quantui import load_result
        data = load_result(Path(path_str))
        past_output.append_display_data(HTML(_format_past_result(data)))
    except Exception as exc:
        past_output.append_stdout(f"Could not load result: {exc}\n")


def _open_results_folder(btn):
    '''Open the results directory in the system file manager (best-effort).'''
    p = _get_results_dir()
    p.mkdir(parents=True, exist_ok=True)
    try:
        if os.name == "nt":
            os.startfile(str(p))                   # type: ignore[attr-defined]
        else:
            subprocess.Popen(["xdg-open", str(p)],
                             stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    except Exception:
        pass  # path label always shows the location as fallback


past_dd.observe(_load_past_result, names="value")
past_refresh_btn.on_click(lambda _: _refresh_results_browser())
open_folder_btn.on_click(_open_results_folder)

# Populate on startup and make the real function visible to _do_run in Cell 3.
_refresh_results_browser()

# ── History panel (assembled widget for the History tab) ─────────────────────
history_panel = widgets.VBox([
    widgets.HTML(
        '<p style="color:#555;font-size:13px;margin:0 0 8px">'
        "Calculations are saved automatically. Select one below to view its results.</p>"
    ),
    widgets.HBox(
        [past_dd, past_refresh_btn, open_folder_btn],
        layout=widgets.Layout(align_items="center", gap="8px"),
    ),
    results_path_lbl,
    past_output,
])


In [ ]:
_advanced_content = widgets.VBox([
    widgets.HTML(
        '<h3 style="margin:8px 0 6px">Compare Results</h3>'
        '<p style="color:#555;font-size:13px;margin:0 0 8px">'
        "Run multiple calculations then add each to compare energies side-by-side.</p>"
    ),
    widgets.HBox([accumulate_btn, clear_btn]),
    comparison_output,
    widgets.HTML('<div style="height:16px"></div>'),
    widgets.HTML(
        '<h3 style="margin:14px 0 6px">Export Standalone Script</h3>'
        '<p style="color:#555;font-size:13px;margin:0 0 8px">'
        "Download a self-contained PySCF script you can study or run outside the notebook.</p>"
    ),
    widgets.HBox([export_btn, export_status]),
])

advanced_accordion = widgets.Accordion(children=[_advanced_content])
advanced_accordion.set_title(0, "Advanced — Compare & Export")
advanced_accordion.selected_index = None  # collapsed by default


In [ ]:
from quantui.help_content import HELP_TOPICS

# ── Help tab content ──────────────────────────────────────────────────────────
_help_keys   = list(HELP_TOPICS.keys())
_help_labels = [HELP_TOPICS[k]["title"] for k in _help_keys]

help_topic_dd = widgets.Dropdown(
    options=list(zip(_help_labels, _help_keys)),
    description="Topic:",
    style={"description_width": "60px"},
    layout=widgets.Layout(width="460px"),
)
help_content_html = widgets.HTML()

def _render_help_topic(change=None):
    key = help_topic_dd.value
    if key and key in HELP_TOPICS:
        entry = HELP_TOPICS[key]
        help_content_html.value = (
            f'<div style="border:1px solid #e2e8f0;border-radius:6px;'
            f'padding:14px 18px;margin:8px 0;background:#f8fafc;max-width:700px">'
            f'<h4 style="margin:0 0 10px;color:#1e293b;font-size:15px;font-weight:700">'
            f'{entry["title"]}</h4>'
            f'<div style="font-size:14px;color:#334155;line-height:1.6">'
            f'{entry["body"]}</div>'
            f'</div>'
        )

help_topic_dd.observe(_render_help_topic, names="value")
_render_help_topic()  # render first topic on load

help_tab_panel = widgets.VBox([
    widgets.HTML(
        '<p style="color:#555;font-size:13px;margin:4px 0 12px">'
        "Browse help topics below. Click <b>?</b> next to the Method or Basis Set "
        "dropdown in the Calculate tab to jump directly to a relevant topic.</p>"
    ),
    help_topic_dd,
    help_content_html,
], layout=widgets.Layout(padding="8px 0"))

# ── Assemble root tab ─────────────────────────────────────────────────────────
_calculate_content = widgets.VBox([
    step_progress.widget,
    mol_input_container,
    calc_setup_panel,
    run_panel,
    results_panel,
    advanced_accordion,
], layout=widgets.Layout(padding="8px 0"))

root_tab = widgets.Tab(children=[
    _calculate_content,
    history_panel,
    help_tab_panel,
])
root_tab.set_title(0, "Calculate")
root_tab.set_title(1, "History")
root_tab.set_title(2, "Help")

# ── "?" button callbacks: jump to Help tab with specific topic ────────────────
def _show_help_topic(topic: str) -> None:
    if topic in HELP_TOPICS:
        help_topic_dd.value = topic
    root_tab.selected_index = 2

# Overwrite Cell-3 placeholder so "?" buttons find the real function.
globals()["_show_help_topic"] = _show_help_topic

# ── Single root display call ──────────────────────────────────────────────────
display(root_tab)
